# Task 10 Chunking Validation

**Description:** Generate the approved chunks. Verify every chunk has the required category rows. Verify positive/negative/masked targets. Confirm no positive annotations are lost. Validate IDs, locations, hashes, boundaries, configuration, and output order.

Task 9 compared multiple chunking methods using span coverage, boundary splits masking extra chunks, and processing cost. The selected chunking configuration is used here to generate the chunks
that will be used by downstream modeling.

In this section generates the approved chunks from the training documents while preserving contract IDs, context group IDs, chunk IDs, chunk numbers, character boundaries, and token counts.

In [21]:
import os
os.environ["USE_TF"] = "0"  # don't search for TensorFlow (faster import)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # hide the "PyTorch not found" notice


In [2]:
import hashlib
import json
import platform
import re
from pathlib import Path

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

OUTPUT_DIR = Path("task10_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
validation_checks = []

def add_check(section, check, expected, actual, passed=None, note=""):
    """Record one validation check. passed defaults to actual == expected."""
    if passed is None:
        passed = (actual == expected)
    validation_checks.append({
        "section": section,
        "check": check,
        "expected": expected if isinstance(expected, str) else (expected.item() if hasattr(expected, "item") else expected),
        "actual": actual if isinstance(actual, str) else (actual.item() if hasattr(actual, "item") else actual),
        "passed": bool(passed),
        "note": note,
    })

def show_checks(section):
    """Display the checks logged for one section."""
    df = pd.DataFrame(validation_checks)
    df = df[df["section"] == section].reset_index(drop=True)
    display(df)
    n_fail = int((~df["passed"]).sum())
    print(f"{section}: {len(df) - n_fail}/{len(df)} checks passed" + ("" if n_fail == 0 else f"  <-- {n_fail} FAILED"))

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

## Load Frozen Chunking Configuration

In [4]:
# Frozen chunking configuration (from Task 9)

# Task 9's recommendation was paragraph-based chunking:
# 97.14% span coverage, 2.86% boundary splits, 10,841 chunks.

SUPPORTED_METHODS = ("paragraph_based", "512_tokens_128_overlap")
TASK9_CONFIG_PATH = Path("task9_outputs/chunking_config.json")

# To validate the 512/128 sliding window change "method" to "512_tokens_128_overlap"
# overlap=128, long_paragraph_overlap=None

DEFAULT_CONFIG = {
    "method": "paragraph_based",
    "tokenizer_name": "distilroberta-base",
    "chunk_size": 512,               # max model tokens incl. special tokens
    "overlap": None,                 # used only by sliding-window methods
    "long_paragraph_overlap": 64,    # used only when one paragraph > chunk size
    "split": "train",
}

if TASK9_CONFIG_PATH.exists():
    chunking_config = json.loads(TASK9_CONFIG_PATH.read_text())
    config_source = str(TASK9_CONFIG_PATH)
else:
    chunking_config = dict(DEFAULT_CONFIG)
    config_source = "Task 9 recommendation (hard-coded in this notebook)"
# A fingerprint of the config: two runs with the same hash used the same settings.
config_hash = sha256_text(json.dumps(chunking_config, sort_keys=True))

# What Task 9 reported. Used to confirm we reproduce the same results.
TASK9_REFERENCE = {
    "train_documents": 325,
    "train_annotation_sets": 13325,
    "train_spans": 8964,
    "positive_annotation_sets": 1536,
    "valid_evaluation_spans": 3078,
    "methods": {
        "paragraph_based":        {"total_chunks": 10841, "fully_covered_spans": 2990, "boundary_splits": 88},
        "512_tokens_128_overlap": {"total_chunks": 11132, "fully_covered_spans": 2983, "boundary_splits": 95},
    },
}
# Reference numbers for the chosen method (None if the method has no reference).
method_reference = TASK9_REFERENCE["methods"].get(chunking_config["method"])

print("Config source:", config_source)
print(json.dumps(chunking_config, indent=2))
print("Config hash:", config_hash)


Config source: Task 9 recommendation (hard-coded in this notebook)
{
  "method": "paragraph_based",
  "tokenizer_name": "distilroberta-base",
  "chunk_size": 512,
  "overlap": null,
  "long_paragraph_overlap": 64,
  "split": "train"
}
Config hash: 92f302d31bc75704a82735667a3e08f504eccfb3e52078e02f41970c335dd77f


In [5]:
# Load the normalized tables and the frozen split (same source as Task 9)

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "Break-Through-Tech/Accenture-1O-contract-review-challenge/"
    "main/"
)

documents = pd.read_parquet(BASE_URL + "notebooks/split_data/documents.parquet")
annotation_sets = pd.read_parquet(BASE_URL + "notebooks/split_data/annotation_sets.parquet")
spans = pd.read_parquet(BASE_URL + "notebooks/split_data/spans.parquet")
categories = pd.read_parquet(BASE_URL + "notebooks/split_data/categories.parquet")
split_map = pd.read_parquet(BASE_URL + "notebooks/frozen-split_data/frozen-split.parquet")

# The 10 categories selected in Task 5. This ORDER is the fixed for every chunk.
selected_categories = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

n_cats = len(selected_categories)

## Keep TRAINING data only (validation data must never be chunked here)
 Attach the split label to each document; sort so the output order is deterministic.


In [6]:
train_documents = (
    documents.merge(split_map, on="context_group_id", how="left")
    .query("split == 'train'")
    .sort_values("contract_id")
    .reset_index(drop=True)
)
train_contract_ids = set(train_documents["contract_id"])

# Annotation sets and spans that belong to training contracts
train_annotation_sets = annotation_sets[annotation_sets["contract_id"].isin(train_contract_ids)]
train_spans = spans[spans["annotation_set_id"].isin(train_annotation_sets["annotation_set_id"])]

# "Annotated" = the category is really present in the contract (is_impossible == False)
selected_annotation_sets = train_annotation_sets[train_annotation_sets["category_id"].isin(selected_categories)]
positive_annotation_sets = selected_annotation_sets[selected_annotation_sets["is_impossible"] == False]

# Spans for those annotations. We keep the raw count so we can prove that
# dropping rows with missing offsets did not silently remove anything.
positive_spans_raw = train_spans[
    train_spans["annotation_set_id"].isin(positive_annotation_sets["annotation_set_id"])
]

# Valid spans = rows with usable start/end offsets (same construction as Task 9)
evaluation_spans = positive_spans_raw.dropna(subset=["answer_start", "answer_end"]).copy()
evaluation_spans[["answer_start", "answer_end"]] = evaluation_spans[["answer_start", "answer_end"]].astype(int)
evaluation_spans = evaluation_spans.merge(
    positive_annotation_sets[["annotation_set_id", "contract_id", "category_id"]],
    on="annotation_set_id", how="left",
).reset_index(drop=True)

print("Training documents:", len(train_documents))
print("Positive annotation sets:", len(positive_annotation_sets))
print("Valid positive spans:", len(evaluation_spans))

Training documents: 325
Positive annotation sets: 1536
Valid positive spans: 3078


# Load the tokenizer named in the frozen config

In [7]:
tokenizer = AutoTokenizer.from_pretrained(chunking_config["tokenizer_name"], use_fast=True)
num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

S = "1_config_and_data"
add_check(S, "config_method_supported",
          "paragraph_based or 512_tokens_128_overlap", chunking_config["method"],
          passed=chunking_config["method"] in ("paragraph_based", "512_tokens_128_overlap"))
add_check(S, "tokenizer_is_fast", True, bool(tokenizer.is_fast))
add_check(S, "selected_categories_count", 10, len(selected_categories))
add_check(S, "selected_categories_unique", 10, len(set(selected_categories)))
add_check(S, "selected_categories_exist_in_categories_table", 0,
          len(set(selected_categories) - set(categories["category_id"])))
add_check(S, "train_documents_vs_task9", TASK9_REFERENCE["train_documents"], len(train_documents))
add_check(S, "train_annotation_sets_vs_task9", TASK9_REFERENCE["train_annotation_sets"], len(train_annotation_sets))
add_check(S, "train_spans_vs_task9", TASK9_REFERENCE["train_spans"], len(train_spans))
add_check(S, "positive_annotation_sets_vs_task9", TASK9_REFERENCE["positive_annotation_sets"], len(positive_annotation_sets))
add_check(S, "valid_positive_spans_vs_task9", TASK9_REFERENCE["valid_evaluation_spans"], len(evaluation_spans))
add_check(S, "no_validation_documents_in_train", 0,
          int((~train_documents["split"].eq("train")).sum()))
show_checks(S)

,section,check,expected,actual,passed,note
0,1_config_and_data,config_method_supported,paragraph_based or 512_tokens_128_overlap,paragraph_based,True,
1,1_config_and_data,tokenizer_is_fast,True,True,True,
2,1_config_and_data,selected_categories_count,10,10,True,
3,1_config_and_data,selected_categories_unique,10,10,True,
4,1_config_and_data,selected_categories_exist_in_categories_table,0,0,True,
5,1_config_and_data,train_documents_vs_task9,325,325,True,
6,1_config_and_data,train_annotation_sets_vs_task9,13325,13325,True,
7,1_config_and_data,train_spans_vs_task9,8964,8964,True,
8,1_config_and_data,positive_annotation_sets_vs_task9,1536,1536,True,
9,1_config_and_data,valid_positive_spans_vs_task9,3078,3078,True,


1_config_and_data: 11/11 checks passed


## Generate Approved Chunks

Each chunk is a range of characters `[char_start, char_end)` in a contract. The tokenizer reports where every token sits in the text, which is how token windows are converted into character ranges.
Two methods are supported. They use the same logic Task 9 evaluated:
- **Paragraph-based:** pack whole paragraphs into a chunk until the token limit is reached. A single paragraph that is too long is split into token windows with a small overlap.
- **Sliding window:** fixed windows of tokens that overlap, e.g. 512 tokens with 128 overlap.

Each chunk also stores its text and SHA-256 hashes, which Section 6 later verifies.

In [8]:
def make_chunk_record(method, contract_id, group_id, chunk_number, char_start, char_end,n_content_tokens):
    """One chunk row. num_tokens includes the special tokens (<s>, </s>)."""
    return {
        "method": method,
        "contract_id": contract_id,
        "context_group_id": group_id,
        "chunk_id": f"{contract_id}__chunk_{chunk_number:04d}",
        "chunk_number": chunk_number,
        "char_start": int(char_start),
        "char_end": int(char_end),
        "num_tokens": int(n_content_tokens + num_special_tokens),
    }


def create_sliding_chunks(documents_df, chunk_size, overlap, method_name):
    """
    Split each contarct into overlapping token windows.
    Each window holds (chunk size- special tokens) content tokens and the next window starts (content size - overlapt) tokens later
    """

    content_size = chunk_size - num_special_tokens
    step = content_size - overlap
    records = []

    for _, row in documents_df.iterrows():
        encoded = tokenizer(row["context"], add_special_tokens=False, truncation=False, return_offsets_mapping=True)
        offsets = encoded["offset_mapping"]           # (char_start, char_end) per token
        n_tokens = len(encoded["input_ids"])
        contract_records = []

        for start in range(0, n_tokens, step):
            end = min(start + content_size, n_tokens)
            window = offsets[start:end]
            if not window:
                continue
            contract_records.append(make_chunk_record(
                method_name, row["contract_id"], row["context_group_id"],
                len(contract_records), window[0][0], window[-1][1], end - start))
            if end == n_tokens:                       # reached the end of the contract
                break

        records.extend(contract_records)
    return pd.DataFrame(records)


def create_paragraph_chunks(documents_df, chunk_size, long_paragraph_overlap, method_name):
    """
    Pack whole paragraphs into chunks without exceeding the token limit.

    Paragraphs are blocks separated by blank lines. Their original character
    positions in the contract are kept. If one paragraph alone is longer than
    the limit, it is split into token windows with a small overlap.
    """

    max_content = chunk_size - num_special_tokens
    records = []

    for _, row in documents_df.iterrows():
        text = str(row["context"])
        contract_records = []

        def emit(char_start, char_end, n_content_tokens):
            """Save one finished chunk for this contract."""
            contract_records.append(make_chunk_record(
                method_name, row["contract_id"], row["context_group_id"],
                len(contract_records), char_start, char_end, n_content_tokens))

        paragraphs = list(re.finditer(r"\S(?:.*?)(?=\n\s*\n|\Z)", text, flags=re.DOTALL))
        current_start, current_end, current_tokens = None, None, 0   # chunk being built

        for para in paragraphs:
            encoded = tokenizer(para.group(0), add_special_tokens=False, truncation=False, return_offsets_mapping=True)
            para_offsets = encoded["offset_mapping"]
            para_tokens = len(encoded["input_ids"])

            # Case 1: this paragraph alone is too long -> save what we have, then window it
            if para_tokens > max_content:
                if current_start is not None:
                    emit(current_start, current_end, current_tokens)
                    current_start, current_end, current_tokens = None, None, 0
                step = max_content - long_paragraph_overlap
                for start in range(0, para_tokens, step):
                    end = min(start + max_content, para_tokens)
                    piece = para_offsets[start:end]
                    if not piece:
                        continue
                    emit(para.start() + piece[0][0], para.start() + piece[-1][1], end - start)
                    if end == para_tokens:
                        break
                continue

            # Case 2: the paragraph still fits in the current chunk -> add it
            if current_tokens + para_tokens <= max_content:
                if current_start is None:
                    current_start = para.start()
                current_end = para.end()
                current_tokens += para_tokens

            # Case 3: it would overflow -> save the current chunk, start a new one
            else:
                emit(current_start, current_end, current_tokens)
                current_start, current_end, current_tokens = para.start(), para.end(), para_tokens

        if current_start is not None:                 # save the last unfinished chunk
            emit(current_start, current_end, current_tokens)

        records.extend(contract_records)
    return pd.DataFrame(records)


CHUNK_COLUMNS = ["method", "contract_id", "context_group_id", "chunk_id", "chunk_number",
                 "char_start", "char_end", "num_tokens", "chunk_text", "chunk_hash", "document_hash"]


def generate_approved_chunks(documents_df, config):
    """Run the configured chunking method, sort the output, and add chunk text + hashes."""
    if config["method"] == "paragraph_based":
        chunks_df = create_paragraph_chunks(
            documents_df, config["chunk_size"], config["long_paragraph_overlap"], config["method"])
    elif config["method"] == "512_tokens_128_overlap":
        chunks_df = create_sliding_chunks(
            documents_df, config["chunk_size"], config["overlap"], config["method"])
    else:
        raise ValueError(f"Unsupported chunking method: {config['method']}")

    chunks_df = chunks_df.sort_values(["contract_id", "chunk_number"]).reset_index(drop=True)

    text_of = dict(zip(documents_df["contract_id"], documents_df["context"]))
    chunks_df["chunk_text"] = [text_of[c][s:e] for c, s, e in
                               zip(chunks_df["contract_id"], chunks_df["char_start"], chunks_df["char_end"])]
    chunks_df["chunk_hash"] = chunks_df["chunk_text"].map(sha256_text)
    chunks_df["document_hash"] = chunks_df["contract_id"].map({c: sha256_text(t) for c, t in text_of.items()})
    return chunks_df[CHUNK_COLUMNS]

In [9]:
# Generate the approved chunks from the training documents
chunks = generate_approved_chunks(train_documents, chunking_config)

chunks_per_contract = chunks.groupby("contract_id").size()
print("Method:", chunks["method"].unique())
print("Total chunks:", len(chunks))
print("Contracts with chunks:", chunks["contract_id"].nunique())
print("Chunks per contract (mean / median / max):",
      round(chunks_per_contract.mean(), 2), chunks_per_contract.median(), chunks_per_contract.max())
print("Tokens per chunk (mean / max):", round(chunks["num_tokens"].mean(), 1), chunks["num_tokens"].max())

display(chunks.drop(columns=["chunk_text"]).head())

S = "2_generate_chunks"
ref = TASK9_REFERENCE["methods"].get(chunking_config["method"])
add_check(S, "total_chunks_vs_task9", ref["total_chunks"] if ref else "n/a", len(chunks),
          passed=(len(chunks) == ref["total_chunks"]) if ref else True)
add_check(S, "every_train_contract_has_chunks", len(train_documents), chunks["contract_id"].nunique())
add_check(S, "no_empty_chunk_text", 0, int((chunks["chunk_text"].str.strip() == "").sum()))
show_checks(S)

Method: <ArrowStringArray>
['paragraph_based']
Length: 1, dtype: str
Total chunks: 10841
Contracts with chunks: 325
Chunks per contract (mean / median / max): 33.36 21.0 260
Tokens per chunk (mean / max): 380.5 512


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_hash,document_hash
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...


,section,check,expected,actual,passed,note
0,2_generate_chunks,total_chunks_vs_task9,10841,10841,True,
1,2_generate_chunks,every_train_contract_has_chunks,325,325,True,
2,2_generate_chunks,no_empty_chunk_text,0,0,True,


2_generate_chunks: 3/3 checks passed


# Generate the approved chunks from the training documents

In [10]:
chunks = generate_approved_chunks(train_documents, chunking_config)

per_contract = chunks.groupby("contract_id").size()
print("Total chunks:", len(chunks))
print("Chunks per contract (mean / median / max):",
      round(per_contract.mean(), 2), per_contract.median(), per_contract.max())
print("Tokens per chunk (mean / max):", round(chunks["num_tokens"].mean(), 1), chunks["num_tokens"].max())
display(chunks.drop(columns=["chunk_text"]).head())

S = "2_generate_chunks"
if method_reference:
    add_check(S, "total_chunks_vs_task9", method_reference["total_chunks"], len(chunks))
add_check(S, "every_train_contract_has_chunks", len(train_documents), chunks["contract_id"].nunique())
add_check(S, "empty_chunk_texts", 0, int((chunks["chunk_text"].str.strip() == "").sum()))
show_checks(S)

Total chunks: 10841
Chunks per contract (mean / median / max): 33.36 21.0 260
Tokens per chunk (mean / max): 380.5 512


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_hash,document_hash
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...


,section,check,expected,actual,passed,note
0,2_generate_chunks,total_chunks_vs_task9,10841,10841,True,
1,2_generate_chunks,every_train_contract_has_chunks,325,325,True,
2,2_generate_chunks,no_empty_chunk_text,0,0,True,
3,2_generate_chunks,total_chunks_vs_task9,10841,10841,True,
4,2_generate_chunks,every_train_contract_has_chunks,325,325,True,
5,2_generate_chunks,empty_chunk_texts,0,0,True,


2_generate_chunks: 6/6 checks passed


## Validate Required Chunk-Category Rows
Every chunk must have exactly one row for each of the 10 categories, in the fixed order from Section 1: all 10 rows for chunk 1, then all 10 for chunk 2, and so on. The targets are filled in
during Section 4.

In [11]:
# Repeat each chunk 10 times (chunk-major), then cycle through the 10 categories.
chunk_category_rows = (
    chunks[["chunk_id", "contract_id", "context_group_id", "chunk_number"]]
    .loc[chunks.index.repeat(n_cats)]
    .reset_index(drop=True)
)
chunk_category_rows["category_id"] = np.tile(selected_categories, len(chunks))

# Each training contract needs an annotation set for every selected category,
# otherwise there would be no annotation to derive a target from.
existing_pairs = set(zip(selected_annotation_sets["contract_id"], selected_annotation_sets["category_id"]))
missing_pairs = [(c, k) for c in train_contract_ids for k in selected_categories
                 if (c, k) not in existing_pairs]

S = "3_required_rows"
add_check(S, "total_rows_equals_chunks_x_10", len(chunks) * n_cats, len(chunk_category_rows))
add_check(S, "chunks_with_all_10_categories", len(chunks),
          int((chunk_category_rows.groupby("chunk_id")["category_id"].nunique() == n_cats).sum()))
add_check(S, "duplicate_chunk_category_rows", 0,
          int(chunk_category_rows.duplicated(["chunk_id", "category_id"]).sum()))
add_check(S, "categories_outside_selected_10", 0,
          int((~chunk_category_rows["category_id"].isin(selected_categories)).sum()))
add_check(S, "row_chunk_ids_not_in_chunks", 0,
          int((~chunk_category_rows["chunk_id"].isin(chunks["chunk_id"])).sum()))
add_check(S, "train_contract_category_pairs_missing_annotation_set", 0, len(missing_pairs))
show_checks(S)

,section,check,expected,actual,passed,note
0,3_required_rows,total_rows_equals_chunks_x_10,108410,108410,True,
1,3_required_rows,chunks_with_all_10_categories,10841,10841,True,
2,3_required_rows,duplicate_chunk_category_rows,0,0,True,
3,3_required_rows,categories_outside_selected_10,0,0,True,
4,3_required_rows,row_chunk_ids_not_in_chunks,0,0,True,
5,3_required_rows,train_contract_category_pairs_missing_annotati...,0,0,True,


3_required_rows: 6/6 checks passed


## Validate Positive, Negative, and Masked Targets

`classify_chunk` decides one target using plain comparisons:

1. Category not annotated in this contract -> `negative`.
2. Annotated, but no span has usable offsets -> `masked` (the location is unknown).
3. Otherwise compare the chunk's character range with each annotated span:
   - the chunk **contains** the whole span -> counts toward `positive`
   - the chunk **overlaps** the span but doesn't contain it -> counts toward `masked`
   - no overlap -> nothing

Any fully contained span makes the row `positive`; otherwise any partial overlap makes it `masked`; otherwise `negative`.

To make sure this is right, the targets are then re-derived a second way, starting from the spans instead of the chunks, and compared row by row.

In [12]:
def classify_chunk(chunk_start, chunk_end, is_annotated, span_ranges):
    """
    Return (target, n_contained, n_partial) for one chunk and one category.

    chunk_start, chunk_end : character range of the chunk
    is_annotated           : does this contract have a positive annotation for the category?
    span_ranges            : list of (start, end) for that annotation's spans
    """
    if not is_annotated:
        return "negative", 0, 0
    if not span_ranges:                      # annotated, but the location is unknown
        return "masked", 0, 0

    n_contained = n_partial = 0
    for span_start, span_end in span_ranges:
        if chunk_start <= span_start and chunk_end >= span_end:
            n_contained += 1                 # whole span is inside the chunk
        elif chunk_start < span_end and chunk_end > span_start:
            n_partial += 1                   # only part of the span is inside

    if n_contained:
        return "positive", n_contained, n_partial
    if n_partial:
        return "masked", n_contained, n_partial
    return "negative", n_contained, n_partial


# Lookups used by classify_chunk
annotated_pairs = set(zip(positive_annotation_sets["contract_id"], positive_annotation_sets["category_id"]))
span_ranges_by_pair = {
    pair: list(zip(g["answer_start"], g["answer_end"]))
    for pair, g in evaluation_spans.groupby(["contract_id", "category_id"])
}
# Annotated categories where NO span has usable offsets (exceptions to document)
unlocated_pairs = sorted(annotated_pairs - set(span_ranges_by_pair))

# Label every chunk-category row
results = [
    classify_chunk(cs, ce, (contract, cat) in annotated_pairs, span_ranges_by_pair.get((contract, cat), []))
    for contract, cat, cs, ce in zip(
        chunk_category_rows["contract_id"],
        chunk_category_rows["category_id"],
        np.repeat(chunks["char_start"].to_numpy(), n_cats),
        np.repeat(chunks["char_end"].to_numpy(), n_cats),
    )
]
chunk_category_rows[["target", "n_contained_spans", "n_partial_spans"]] = pd.DataFrame(results)

# Convenience columns for downstream modeling
chunk_category_rows["label"] = chunk_category_rows["target"].map({"positive": 1, "negative": 0}).astype("Int8")  # masked -> missing
chunk_category_rows["is_masked"] = chunk_category_rows["target"] == "masked"
chunk_category_rows["annotation_present"] = [
    pair in annotated_pairs for pair in zip(chunk_category_rows["contract_id"], chunk_category_rows["category_id"])
]

print("Annotated categories with no usable span offsets:", len(unlocated_pairs))
print(chunk_category_rows["target"].value_counts())

Annotated categories with no usable span offsets: 0
target
negative    105869
positive      2324
masked         217
Name: count, dtype: int64


In [13]:
# Target counts per category
target_summary = (
    chunk_category_rows.pivot_table(index="category_id", columns="target", values="chunk_id", aggfunc="count", fill_value=0).reindex(selected_categories).reindex(columns=["positive", "masked","negative"], fill_value=0)
)
target_summary["total"] = target_summary.sum(axis=1)
target_summary["masked_pct"] = (target_summary["masked"] / target_summary["total"] * 100).round(2)
display(target_summary)

target,positive,masked,negative,total,masked_pct
category_id,,,,,
governing_law,291,11,10539,10841,0.10
renewal_term,127,5,10709,10841,0.05
revenue_profit_sharing,178,35,10628,10841,0.32
cap_on_liability,333,52,10456,10841,0.48
uncapped_liability,91,18,10732,10841,0.17
termination_for_convenience,143,8,10690,10841,0.07
anti_assignment,313,19,10509,10841,0.18
audit_rights,259,13,10569,10841,0.12
license_grant,386,38,10417,10841,0.35


### Independent re-derivation: SPAN-driven (loop over every span, then every chunk of its contract). The first pass above was chunk-driven and vectorized, so agreement between them is a real cross-check.

In [14]:
def audit_spans(chunks_df, spans_df):
    """
    Independent check: start from each SPAN and look at every chunk of its contract.

    Returns
      span_audit         : one row per span with how many chunks contain / touch it
      expected_positive  : set of (chunk_id, category_id) that must be positive
      expected_masked    : set of (chunk_id, category_id) that must be masked
    """
    chunks_of = {c: g for c, g in chunks_df.groupby("contract_id")}
    audit_rows, expected_positive, expected_masked = [], set(), set()

    for sp in spans_df.itertuples(index=False):
        n_containing = n_touching = 0
        for ch in chunks_of.get(sp.contract_id, pd.DataFrame()).itertuples(index=False):
            contains = ch.char_start <= sp.answer_start and ch.char_end >= sp.answer_end
            touches = ch.char_start < sp.answer_end and ch.char_end > sp.answer_start
            if contains:
                expected_positive.add((ch.chunk_id, sp.category_id))
                n_containing += 1
            elif touches:
                expected_masked.add((ch.chunk_id, sp.category_id))
            n_touching += contains or touches

        audit_rows.append({
            "span_id": sp.span_id, "annotation_set_id": sp.annotation_set_id,
            "contract_id": sp.contract_id, "category_id": sp.category_id,
            "answer_start": sp.answer_start, "answer_end": sp.answer_end,
            "n_chunks_fully_containing": n_containing, "n_chunks_touching": int(n_touching),
        })
    return pd.DataFrame(audit_rows), expected_positive, expected_masked


span_audit, expected_positive, expected_masked = audit_spans(chunks, evaluation_spans)

# Annotations without usable offsets are masked in every chunk of their contract
for contract, cat in unlocated_pairs:
    expected_masked |= {(chunk_id, cat) for chunk_id in chunks.loc[chunks["contract_id"] == contract, "chunk_id"]}
expected_masked -= expected_positive       # positive wins over masked


def pairs_with_target(target):
    """Set of (chunk_id, category_id) rows carrying the given target."""
    rows = chunk_category_rows[chunk_category_rows["target"] == target]
    return set(zip(rows["chunk_id"], rows["category_id"]))


t = chunk_category_rows["target"]
S = "4_targets"
add_check(S, "rows_with_invalid_target", 0, int((~t.isin(["positive", "negative", "masked"])).sum()))
add_check(S, "label_inconsistent_with_target", 0, int(
    ((t == "positive") & (chunk_category_rows["label"] != 1)).sum()
    + ((t == "negative") & (chunk_category_rows["label"] != 0)).sum()
    + ((t == "masked") & chunk_category_rows["label"].notna()).sum()))
add_check(S, "positive_or_masked_rows_without_annotation", 0,
          int(((t != "negative") & ~chunk_category_rows["annotation_present"]).sum()),
          note="Categories with is_impossible == True must be all-negative.")
add_check(S, "positive_rows_differing_from_independent_recompute", 0,
          len(pairs_with_target("positive") ^ expected_positive))
add_check(S, "masked_rows_differing_from_independent_recompute", 0,
          len(pairs_with_target("masked") ^ expected_masked))
add_check(S, "annotated_categories_without_usable_offsets", 0, len(unlocated_pairs),
          note="Any nonzero value is an exception that must be documented.")
show_checks(S)

,section,check,expected,actual,passed,note
0,4_targets,rows_with_invalid_target,0,0,True,
1,4_targets,label_inconsistent_with_target,0,0,True,
2,4_targets,positive_or_masked_rows_without_annotation,0,0,True,Categories with is_impossible == True must be ...
3,4_targets,positive_rows_differing_from_independent_recom...,0,0,True,
4,4_targets,masked_rows_differing_from_independent_recompute,0,0,True,
5,4_targets,annotated_categories_without_usable_offsets,0,0,True,Any nonzero value is an exception that must be...


4_targets: 6/6 checks passed


## 5. Verify No Positive Annotations Are Lost
"Lost" is checked at three levels:

1. **Span level:** every annotated span must overlap at least one chunk (none should fall into a gap between chunks).
2. **Row level:** no chunk that overlaps a positive span should be labeled `negative` for that category.
3. **Annotation level:** every positive annotation set must have at least one `positive` chunk.

Spans cut across a chunk boundary (Task 9's "boundary splits") are not lost. They are `masked` in the chunks that hold pieces of them. They are listed below as documented exceptions.

In [15]:
# Span status: fully contained in a chunk / cut by a boundary / touching no chunk at all
span_audit["status"] = np.select(
    [span_audit["n_chunks_fully_containing"] > 0, span_audit["n_chunks_touching"] > 0],
    ["fully_contained", "boundary_split"],
    default="LOST",
)
print(span_audit["status"].value_counts())

# Annotation level: does every positive annotation set have at least one positive chunk?
positive_counts = (chunk_category_rows[chunk_category_rows["target"] == "positive"]
                   .groupby(["contract_id", "category_id"]).size())
positive_sets_check = positive_annotation_sets[["annotation_set_id", "contract_id", "category_id"]].copy()
positive_sets_check["n_positive_chunks"] = [
    int(positive_counts.get(pair, 0))
    for pair in zip(positive_sets_check["contract_id"], positive_sets_check["category_id"])
]
lost_sets = positive_sets_check[positive_sets_check["n_positive_chunks"] == 0]

# Row level: any chunk touching a positive span but labeled negative?
target_of = dict(zip(zip(chunk_category_rows["chunk_id"], chunk_category_rows["category_id"]), t))
negative_but_touching = [pair for pair in (expected_positive | expected_masked) if target_of[pair] == "negative"]

# Per-category preservation table
category_preservation = (
    span_audit.groupby("category_id")["status"].value_counts().unstack(fill_value=0)
    .reindex(selected_categories, fill_value=0)
    .reindex(columns=["fully_contained", "boundary_split", "LOST"], fill_value=0)
)
category_preservation["total_spans"] = category_preservation.sum(axis=1)
category_preservation["coverage_pct"] = (
    category_preservation["fully_contained"] / category_preservation["total_spans"].replace(0, np.nan) * 100).round(2)
display(category_preservation)

n_contained = int((span_audit["status"] == "fully_contained").sum())
n_split = int((span_audit["status"] == "boundary_split").sum())

S = "5_no_positives_lost"
add_check(S, "positive_spans_dropped_for_missing_offsets", 0, len(positive_spans_raw) - len(evaluation_spans))
add_check(S, "spans_touching_no_chunk_LOST", 0, int((span_audit["status"] == "LOST").sum()))
add_check(S, "chunks_touching_positive_span_labeled_negative", 0, len(negative_but_touching))
add_check(S, "positive_annotation_sets_with_no_positive_chunk", 0, len(lost_sets))
add_check(S, "positive_annotation_sets_checked", len(positive_annotation_sets), len(positive_sets_check))
if method_reference:
    add_check(S, "spans_fully_contained_vs_task9", method_reference["fully_covered_spans"], n_contained)
    add_check(S, "boundary_split_spans_vs_task9", method_reference["boundary_splits"], n_split)
show_checks(S)

status
fully_contained    2990
boundary_split       88
Name: count, dtype: int64


status,fully_contained,boundary_split,LOST,total_spans,coverage_pct
category_id,,,,,
governing_law,289,1,0,290,99.66
renewal_term,141,3,0,144,97.92
revenue_profit_sharing,250,19,0,269,92.94
cap_on_liability,438,20,0,458,95.63
uncapped_liability,116,7,0,123,94.31
termination_for_convenience,164,1,0,165,99.39
anti_assignment,401,7,0,408,98.28
audit_rights,412,5,0,417,98.80
license_grant,513,17,0,530,96.79


,section,check,expected,actual,passed,note
0,5_no_positives_lost,positive_spans_dropped_for_missing_offsets,0,0,True,
1,5_no_positives_lost,spans_touching_no_chunk_LOST,0,0,True,
2,5_no_positives_lost,chunks_touching_positive_span_labeled_negative,0,0,True,
3,5_no_positives_lost,positive_annotation_sets_with_no_positive_chunk,0,20,False,
4,5_no_positives_lost,positive_annotation_sets_checked,1536,1536,True,
5,5_no_positives_lost,spans_fully_contained_vs_task9,2990,2990,True,
6,5_no_positives_lost,boundary_split_spans_vs_task9,88,88,True,


5_no_positives_lost: 6/7 checks passed  <-- 1 FAILED


In [16]:
# Documented exceptions: spans cut across a chunk boundary (masked, not lost)
split_spans = span_audit[span_audit["status"] == "boundary_split"].copy()
chunks_of_contract = {c: g for c, g in chunks.groupby("contract_id")}


def best_single_chunk_coverage(span):
    """Percent of the span visible in the single best chunk."""
    best = 0
    for ch in chunks_of_contract[span["contract_id"]].itertuples(index=False):
        overlap = min(span["answer_end"], ch.char_end) - max(span["answer_start"], ch.char_start)
        best = max(best, overlap)
    return round(best / max(span["answer_end"] - span["answer_start"], 1) * 100, 1)


split_spans["best_chunk_coverage_pct"] = [best_single_chunk_coverage(r) for _, r in split_spans.iterrows()]
split_spans["annotation_set_has_positive_chunk"] = split_spans["annotation_set_id"].map(
    positive_sets_check.set_index("annotation_set_id")["n_positive_chunks"] > 0).astype(bool)

print(f"{len(split_spans)} boundary-split spans. "
      f"Their annotation set still has a positive chunk in "
      f"{int(split_spans['annotation_set_has_positive_chunk'].sum())} cases.")
display(split_spans.sort_values("best_chunk_coverage_pct").head(10))

88 boundary-split spans. Their annotation set still has a positive chunk in 67 cases.


,span_id,annotation_set_id,contract_id,category_id,answer_start,answer_end,n_chunks_fully_containing,n_chunks_touching,status,best_chunk_coverage_pct,annotation_set_has_positive_chunk
1188,contract_0167__renewal_term__span_001,contract_0167__renewal_term,contract_0167,renewal_term,11027,13613,0,2,boundary_split,50.0,True
1077,contract_0158__exclusivity__span_000,contract_0158__exclusivity,contract_0158,exclusivity,82567,83540,0,2,boundary_split,50.1,True
2858,contract_0371__revenue_profit_sharing__span_002,contract_0371__revenue_profit_sharing,contract_0371,revenue_profit_sharing,2689,3578,0,3,boundary_split,51.2,True
347,contract_0057__anti_assignment__span_000,contract_0057__anti_assignment,contract_0057,anti_assignment,60509,60840,0,2,boundary_split,52.6,True
2604,contract_0346__revenue_profit_sharing__span_000,contract_0346__revenue_profit_sharing,contract_0346,revenue_profit_sharing,33018,35581,0,3,boundary_split,53.1,False
1573,contract_0209__license_grant__span_002,contract_0209__license_grant,contract_0209,license_grant,36692,37870,0,2,boundary_split,54.3,True
2905,contract_0382__revenue_profit_sharing__span_001,contract_0382__revenue_profit_sharing,contract_0382,revenue_profit_sharing,18140,18323,0,2,boundary_split,54.6,True
942,contract_0137__audit_rights__span_003,contract_0137__audit_rights,contract_0137,audit_rights,149230,150464,0,2,boundary_split,54.9,True
1993,contract_0253__anti_assignment__span_002,contract_0253__anti_assignment,contract_0253,anti_assignment,57902,58233,0,2,boundary_split,55.9,True
322,contract_0047__cap_on_liability__span_005,contract_0047__cap_on_liability,contract_0047,cap_on_liability,46686,47270,0,3,boundary_split,56.3,True


## 6. Validate IDs, Locations, Hashes, and Boundaries

**IDs:** chunk IDs are unique, correctly formatted, and consistent with contract ID and chunk number; each chunk's context group ID matches its document; no validation-split group appears in the chunks.

**Locations:** every chunk range lies inside its document, and every annotated span range lies inside its document and reproduces its `answer_text`.

**Hashes:** chunk and document hashes are recomputed from the original `documents` table (not the copy used to generate them) and must match.

**Boundaries:** chunks are in order, all non-whitespace text is covered by at least one chunk, and each chunk fits the model's 512-token limit when re-tokenized from its own text.

In [17]:
S = "6_ids_locations_hashes_boundaries"
full_text = dict(zip(documents["contract_id"], documents["context"]))       # independent source
full_doc_hash = {c: sha256_text(t) for c, t in full_text.items()}
doc_len = {c: len(t) for c, t in full_text.items()}

# IDs
parsed = chunks["chunk_id"].str.extract(r"^(?P<cid>.+)__chunk_(?P<num>\d{4})$")
well_formed = parsed["cid"].notna()
add_check(S, "chunk_id_duplicates", 0, int(chunks["chunk_id"].duplicated().sum()))
add_check(S, "chunk_id_bad_format", 0, int((~well_formed).sum()))
add_check(S, "chunk_id_prefix_not_matching_contract_id", 0,
          int((parsed.loc[well_formed, "cid"] != chunks.loc[well_formed, "contract_id"]).sum()))
add_check(S, "chunk_id_suffix_not_matching_chunk_number", 0,
          int((parsed.loc[well_formed, "num"].astype(int) != chunks.loc[well_formed, "chunk_number"]).sum()))

group_by_contract = dict(zip(documents["contract_id"], documents["context_group_id"]))
add_check(S, "context_group_id_mismatch_with_documents", 0,
          int((chunks["contract_id"].map(group_by_contract) != chunks["context_group_id"]).sum()))
validation_groups = set(split_map.loc[split_map["split"] == "validation", "context_group_id"])
add_check(S, "chunks_from_validation_context_groups", 0,
          int(chunks["context_group_id"].isin(validation_groups).sum()),
          note="Leakage guard: no validation group may appear in training chunks.")
add_check(S, "chunk_contracts_not_in_train_split", 0,
          int((~chunks["contract_id"].isin(train_contract_ids)).sum()))

# Locations: chunks
chunk_doc_len = chunks["contract_id"].map(doc_len)
add_check(S, "chunk_char_start_negative", 0, int((chunks["char_start"] < 0).sum()))
add_check(S, "chunk_char_end_not_after_start", 0, int((chunks["char_end"] <= chunks["char_start"]).sum()))
add_check(S, "chunk_char_end_beyond_document", 0, int((chunks["char_end"] > chunk_doc_len).sum()))

# Locations: spans
sp = evaluation_spans.copy()
sp_doc_len = sp["contract_id"].map(doc_len)
add_check(S, "span_start_negative", 0, int((sp["answer_start"] < 0).sum()))
add_check(S, "span_end_not_after_start", 0, int((sp["answer_end"] <= sp["answer_start"]).sum()))
add_check(S, "span_end_beyond_document", 0, int((sp["answer_end"] > sp_doc_len).sum()))

def norm_ws(t):
    return " ".join(str(t).split())

sliced = [full_text[c][s:e] for c, s, e in zip(sp["contract_id"], sp["answer_start"], sp["answer_end"])]
exact_mismatch = int(sum(a != b for a, b in zip(sliced, sp["answer_text"])))
norm_mismatch = int(sum(norm_ws(a) != norm_ws(b) for a, b in zip(sliced, sp["answer_text"])))
add_check(S, "span_text_mismatch_after_whitespace_normalization", 0, norm_mismatch,
          note="Confirms answer_start/answer_end (end exclusive) point at answer_text.")
add_check(S, "span_text_exact_mismatch_info_only", 0, exact_mismatch, passed=True,
          note="Informational: nonzero means only whitespace differs.")

# Hashes
recomputed_text = [full_text[c][s:e] for c, s, e in zip(chunks["contract_id"], chunks["char_start"], chunks["char_end"])]
recomputed_hash = [sha256_text(t) for t in recomputed_text]
add_check(S, "chunk_text_differs_from_document_slice", 0,
          int(sum(a != b for a, b in zip(recomputed_text, chunks["chunk_text"]))))
add_check(S, "chunk_hash_mismatch_on_recompute", 0,
          int(sum(a != b for a, b in zip(recomputed_hash, chunks["chunk_hash"]))))
add_check(S, "document_hash_mismatch_on_recompute", 0,
          int((chunks["contract_id"].map(full_doc_hash) != chunks["document_hash"]).sum()))
n_dup_hash = int(chunks["chunk_hash"].duplicated().sum())
add_check(S, "duplicate_chunk_hashes_info_only", 0, n_dup_hash, passed=True,
          note="Informational: identical text in different chunks (boilerplate across contracts).")

# Boundaries: order and coverage
bad_numbering, bad_order, uncovered_nonwhitespace = 0, 0, 0
for contract_id, g in chunks.groupby("contract_id"):
    if g["chunk_number"].tolist() != list(range(len(g))):
        bad_numbering += 1
    if not (g["char_start"].diff().dropna() >= 0).all():
        bad_order += 1
    # Walk the chunks in order; any gap between them must be whitespace only
    text = full_text[contract_id]
    covered_until = 0
    for cs, ce in zip(g["char_start"], g["char_end"]):
        if cs > covered_until and text[covered_until:cs].strip():
            uncovered_nonwhitespace += 1
        covered_until = max(covered_until, ce)
    if text[covered_until:].strip():
        uncovered_nonwhitespace += 1

add_check(S, "contracts_with_non_contiguous_chunk_numbers", 0, bad_numbering)
add_check(S, "contracts_with_chunk_char_start_out_of_order", 0, bad_order)
add_check(S, "uncovered_gaps_containing_text", 0, uncovered_nonwhitespace,
          note="Text between/after chunks that no chunk contains (whitespace gaps are allowed).")

# Boundaries: model token limit

# Re-tokenize each chunk's own text WITH special tokens, exactly as the model will see it.

max_len = chunking_config["chunk_size"]
retokenized = tokenizer(chunks["chunk_text"].tolist(), add_special_tokens=True, truncation=False)
chunks["retokenized_num_tokens"] = [len(ids) for ids in retokenized["input_ids"]]
over_limit = chunks[chunks["retokenized_num_tokens"] > max_len]
understated = int((chunks["retokenized_num_tokens"] > chunks["num_tokens"]).sum())

add_check(S, "recorded_num_tokens_over_limit", 0, int((chunks["num_tokens"] > max_len).sum()))
add_check(S, "retokenized_chunks_over_512_tokens", 0, len(over_limit), note="Chunks the model would silently truncate, which could cut off part of a span.")
add_check(S, "recorded_num_tokens_lower_than_retokenized_info_only", 0, understated, passed=True, note="Informational: paragraph packing sums paragraph tokens and ignores the blank-line tokens between paragraphs.")
print("Max recorded tokens:", chunks["num_tokens"].max(), "| Max re-tokenized tokens:", chunks["retokenized_num_tokens"].max())

show_checks(S)

if len(over_limit):
    print("\nChunks that exceed the model limit once re-tokenized:")
    display(over_limit[["chunk_id", "num_tokens", "retokenized_num_tokens"]].head(20))

Max recorded tokens: 512 | Max re-tokenized tokens: 1035


,section,check,expected,actual,passed,note
0,6_ids_locations_hashes_boundaries,chunk_id_duplicates,0,0,True,
1,6_ids_locations_hashes_boundaries,chunk_id_bad_format,0,0,True,
2,6_ids_locations_hashes_boundaries,chunk_id_prefix_not_matching_contract_id,0,0,True,
3,6_ids_locations_hashes_boundaries,chunk_id_suffix_not_matching_chunk_number,0,0,True,
4,6_ids_locations_hashes_boundaries,context_group_id_mismatch_with_documents,0,0,True,
5,6_ids_locations_hashes_boundaries,chunks_from_validation_context_groups,0,0,True,Leakage guard: no validation group may appear ...
6,6_ids_locations_hashes_boundaries,chunk_contracts_not_in_train_split,0,0,True,
7,6_ids_locations_hashes_boundaries,chunk_char_start_negative,0,0,True,
8,6_ids_locations_hashes_boundaries,chunk_char_end_not_after_start,0,0,True,
9,6_ids_locations_hashes_boundaries,chunk_char_end_beyond_document,0,0,True,


6_ids_locations_hashes_boundaries: 24/25 checks passed  <-- 1 FAILED

Chunks that exceed the model limit once re-tokenized:


,chunk_id,num_tokens,retokenized_num_tokens
0,contract_0001__chunk_0000,463,582
7,contract_0001__chunk_0007,495,578
8,contract_0001__chunk_0008,442,550
34,contract_0001__chunk_0034,481,519
41,contract_0001__chunk_0041,505,552
43,contract_0001__chunk_0043,487,514
52,contract_0001__chunk_0052,486,549
53,contract_0001__chunk_0053,473,547
59,contract_0002__chunk_0001,510,514
60,contract_0002__chunk_0002,497,524


## 7. Validate Chunking Configuration and Output Order
- The chunks were produced by the frozen method, tokenizer, and chunk size.
- Regenerating the chunks from scratch gives **identical** output (the process is deterministic).
- The results reproduce Task 9's numbers for the same method.
- Chunks are sorted by `contract_id`, then `chunk_number`; the category rows follow chunk order, then the fixed category order; the columns match the expected schema.

In [18]:
S = "7_config_and_order"
cfg = chunking_config
ROW_COLUMNS = ["chunk_id", "contract_id", "context_group_id", "chunk_number", "category_id", "target",
               "label", "is_masked", "annotation_present", "n_contained_spans", "n_partial_spans"]

# Configuration
add_check(S, "chunk_method_matches_frozen_config", cfg["method"], ",".join(sorted(chunks["method"].unique())))
add_check(S, "tokenizer_matches_frozen_config", cfg["tokenizer_name"], tokenizer.name_or_path)
add_check(S, "chunk_size_is_512", 512, cfg["chunk_size"])
if cfg["method"] == "512_tokens_128_overlap":
    add_check(S, "overlap_is_128", 128, cfg["overlap"])
    # every window except the last one in each contract should be completely full
    all_full = all((g["num_tokens"].iloc[:-1] == cfg["chunk_size"]).all() for _, g in chunks.groupby("contract_id"))
    add_check(S, "sliding_windows_full_except_last", True, bool(all_full))
else:
    add_check(S, "long_paragraph_overlap_is_64", 64, cfg["long_paragraph_overlap"])

# Determinism: chunk again and compare
chunks_again = generate_approved_chunks(train_documents, chunking_config)
add_check(S, "regenerated_chunks_identical", True, bool(chunks[CHUNK_COLUMNS].equals(chunks_again)),
          note="Chunking the same documents twice gives identical results.")

# Reproduces Task 9
if method_reference:
    add_check(S, "task9_total_chunks_reproduced", method_reference["total_chunks"], len(chunks))

# Output order and schema
expected_chunk_order = chunks[["contract_id", "chunk_number"]].sort_values(["contract_id", "chunk_number"]).reset_index(drop=True)
add_check(S, "chunks_sorted_by_contract_then_chunk_number", True,
          bool(chunks[["contract_id", "chunk_number"]].equals(expected_chunk_order)))
add_check(S, "chunk_columns_match_schema", True, list(chunks[CHUNK_COLUMNS].columns) == CHUNK_COLUMNS)

expected_row_order = pd.DataFrame({
    "chunk_id": np.repeat(chunks["chunk_id"].to_numpy(), n_cats),
    "category_id": np.tile(selected_categories, len(chunks)),
})
add_check(S, "rows_are_chunk_order_then_category_order", True,
          bool(expected_row_order.equals(chunk_category_rows[["chunk_id", "category_id"]])))
add_check(S, "row_columns_match_schema", True, set(ROW_COLUMNS) <= set(chunk_category_rows.columns))
show_checks(S)

,section,check,expected,actual,passed,note
0,7_config_and_order,chunk_method_matches_frozen_config,paragraph_based,paragraph_based,True,
1,7_config_and_order,tokenizer_matches_frozen_config,distilroberta-base,distilroberta-base,True,
2,7_config_and_order,chunk_size_is_512,512,512,True,
3,7_config_and_order,long_paragraph_overlap_is_64,64,64,True,
4,7_config_and_order,regenerated_chunks_identical,True,True,True,Chunking the same documents twice gives identi...
5,7_config_and_order,task9_total_chunks_reproduced,10841,10841,True,
6,7_config_and_order,chunks_sorted_by_contract_then_chunk_number,True,True,True,
7,7_config_and_order,chunk_columns_match_schema,True,True,True,
8,7_config_and_order,rows_are_chunk_order_then_category_order,True,True,True,
9,7_config_and_order,row_columns_match_schema,True,True,True,


7_config_and_order: 10/10 checks passed


## 8. Save Validation Evidence

Save the approved chunks, the chunk-category target rows, the frozen configuration, the documented exceptions, and one machine-readable evidence file. The parquet files are reloaded from disk and compared with what is in memory, then hashed.

**For the independent check:** a second teammate can rerun this notebook and compare `content_sha256` in the evidence file. It is computed from the data itself, so it should match even when the parquet *file* bytes (`md5`) differ because of different pyarrow versions.

In [19]:
S = "8_artifacts"


def file_md5(path):
    """MD5 of the file bytes (can differ across pyarrow versions)."""
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def content_sha256(df):
    """Hash of the data itself, independent of how the parquet file is encoded."""
    return hashlib.sha256(pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values.tobytes()).hexdigest()


def save_and_verify(name, df):
    """Save df as parquet, reload it, check it round-trips, and return its hashes."""
    path = OUTPUT_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    reloaded = pd.read_parquet(path)
    add_check(S, f"{name}_parquet_roundtrip_identical", True, bool(reloaded.equals(df)))
    add_check(S, f"{name}_parquet_row_count", len(df), len(reloaded))
    return {"path": str(path), "rows": int(len(reloaded)), "md5": file_md5(path),
            "content_sha256": content_sha256(reloaded)}


chunks_out = chunks[CHUNK_COLUMNS].copy()           # drops the helper column added during validation
rows_out = chunk_category_rows[ROW_COLUMNS].copy()

artifact_hashes = {
    "chunks": save_and_verify("chunks", chunks_out),
    "chunk_category_rows": save_and_verify("chunk_category_rows", rows_out),
}

# Frozen config and documented exceptions
(OUTPUT_DIR / "chunking_config_frozen.json").write_text(
    json.dumps({**chunking_config, "config_hash": config_hash, "config_source": config_source}, indent=2))
split_spans.to_csv(OUTPUT_DIR / "boundary_split_spans_exceptions.csv", index=False)
target_summary.to_csv(OUTPUT_DIR / "target_summary_by_category.csv")
category_preservation.to_csv(OUTPUT_DIR / "span_preservation_by_category.csv")

show_checks(S)
artifact_hashes

,section,check,expected,actual,passed,note
0,8_artifacts,chunks_parquet_roundtrip_identical,True,True,True,
1,8_artifacts,chunks_parquet_row_count,10841,10841,True,
2,8_artifacts,chunk_category_rows_parquet_roundtrip_identical,True,True,True,
3,8_artifacts,chunk_category_rows_parquet_row_count,108410,108410,True,


8_artifacts: 4/4 checks passed


{'chunks': {'path': 'task10_outputs\\chunks.parquet',
  'rows': 10841,
  'md5': 'ddc04060c46ecbf3d174e49674476ca1',
  'content_sha256': '9341c770986c86bd367b6a2142f6077e7741492ca167b0e289478412ec1cbbab'},
 'chunk_category_rows': {'path': 'task10_outputs\\chunk_category_rows.parquet',
  'rows': 108410,
  'md5': '4f831d4e5785cac46d4ccb1cdb63e861',
  'content_sha256': '8145bf9d091f96aeded159ce7ca144315d4abd7f465a32af89e9da2756bb11d9'}}

# Final Task 10 evidence report

In [20]:
all_checks = pd.DataFrame(validation_checks)
failed = all_checks[~all_checks["passed"]]
task10_passed = failed.empty

evidence = {
    "task": "task10_chunking_validation",
    "task10_passed": bool(task10_passed),
    "n_checks": int(len(all_checks)),
    "n_failed": int(len(failed)),
    "checks": all_checks.to_dict(orient="records"),
    "chunking_config": chunking_config,
    "config_hash": config_hash,
    "config_source": config_source,
    "counts": {
        "train_documents": int(len(train_documents)),
        "chunks": int(len(chunks_out)),
        "chunk_category_rows": int(len(rows_out)),
        "positive_rows": int((rows_out["target"] == "positive").sum()),
        "masked_rows": int((rows_out["target"] == "masked").sum()),
        "negative_rows": int((rows_out["target"] == "negative").sum()),
        "positive_annotation_sets": int(len(positive_annotation_sets)),
        "valid_positive_spans": int(len(evaluation_spans)),
        "spans_fully_contained": n_contained,
        "spans_boundary_split": n_split,
        "spans_lost": int((span_audit["status"] == "LOST").sum()),
        "positive_annotation_sets_lost": int(len(lost_sets)),
    },
    "artifact_hashes": artifact_hashes,
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "numpy": np.__version__, "tokenizer": tokenizer.name_or_path},
}
(OUTPUT_DIR / "chunking_validation_evidence.json").write_text(json.dumps(evidence, indent=2, default=str))

print(f"TASK 10 CHUNKING VALIDATION: {'PASSED' if task10_passed else 'FAILED'}  "
      f"({len(all_checks) - len(failed)}/{len(all_checks)} checks passed)")
if not task10_passed:
    display(failed)
all_checks.groupby("section")["passed"].agg(passed="sum", total="count")

TASK 10 CHUNKING VALIDATION: FAILED  (73/75 checks passed)


,section,check,expected,actual,passed,note
32,5_no_positives_lost,positive_annotation_sets_with_no_positive_chunk,0,20,False,
59,6_ids_locations_hashes_boundaries,retokenized_chunks_over_512_tokens,0,1896,False,"Chunks the model would silently truncate, whic..."


,passed,total
section,,
1_config_and_data,11,11
2_generate_chunks,6,6
3_required_rows,6,6
4_targets,6,6
5_no_positives_lost,6,7
6_ids_locations_hashes_boundaries,24,25
7_config_and_order,10,10
8_artifacts,4,4


## For your independent reviewer

1. Have a teammate - pull this branch and run the notebook fresh in Colab.
2. Compare their `task10_outputs/chunking_validation_evidence.json` with yours, in particular `config_hash`, `counts`, and each artifact's `content_sha256`.
3. Attach both JSON files to the Task 10 PR.
If `md5` differs but `content_sha256` matches, the data is identical and only the parquet encoding differs (different pyarrow version).